# Project 1 — Realized Volatility & Liquidity

IF / IH minute bars → RV (5-min + Garman-Klass) → rolling stats → Amihud / volume-OI profiles.

For a one-shot export of CSVs + figures, prefer:
`python projects/01_volatility_liquidity/run_analysis.py --use-sample`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.sample import write_sample_dataset
from src.data.io import load_minute_bars
from src.data.clean import clean_minute_bars
from src.metrics import (
    daily_realized_variance,
    garman_klass_vol,
    intraday_rv_profile,
    rolling_volatility,
    rolling_correlation,
    amihud_illiquidity,
)

write_sample_dataset()
if_df = clean_minute_bars(load_minute_bars(ROOT / "data/sample/IF_1min.parquet"))
ih_df = clean_minute_bars(load_minute_bars(ROOT / "data/sample/IH_1min.parquet"))
if_df.shape, ih_df.shape

In [ ]:
rv = daily_realized_variance(if_df, sampling_minutes=5)
gk = garman_klass_vol(if_df)
profile = intraday_rv_profile(if_df, sampling_minutes=5)
amihud = amihud_illiquidity(if_df, sampling_minutes=5)

profile["mean_r2"].plot(title="IF intraday RV pattern", figsize=(10, 3))
rv.tail(), gk.tail(), amihud.tail()

In [ ]:
roll = rolling_volatility(if_df, windows=[20, 60])
corr = rolling_correlation(if_df, ih_df, windows=[20, 60])
ax = roll.plot(title="IF rolling vol", figsize=(10, 3))
corr.plot(title="IF–IH rolling corr", figsize=(10, 3))